# Avaliação 2 - Desenvolvimento de Modelos de Deep Learning
**Aluno:** Vitor Fontenele de Oliveira Linhares
**Mátricula:** 1700778

## Setup

In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
from torch.utils.data import DataLoader, TensorDataset
import torchvision
from torchvision.datasets import FashionMNIST, MNIST
from torchvision.utils import make_grid
from ucimlrepo import fetch_ucirepo
import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
try:
    import torch_geometric.nn as geom_nn
except ImportError:
    print("Aviso: 'torch_geometric' não encontrado. Caso não possua, instale para a Questão 4 (GNN).")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Questão 1: MLP - Breast Cancer Wisconsin

### 1) Descrição e entendimento do problema

O dataset reúne atributos numéricos extraídos de imagens de núcleos de células, e a tarefa aqui é prever o diagnóstico entre benigno (B) e maligno (M). Para começar com segurança, fizemos uma exploração inicial para entender o formato da base, observar o comportamento das variáveis e verificar se existiam sinais de desbalanceamento entre as classes e correlações relevantes.

In [ ]:
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 

target_col = y['Diagnosis']
target_numeric = target_col.map({'B': 0, 'M': 1})

df = pd.concat([X, target_numeric.rename('Diagnosis_Target')], axis=1)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

first_feat = X.columns[0]
sns.histplot(data=df, x=first_feat, hue=target_col, kde=True, palette='Set2', ax=axes[0])
axes[0].set_title(f'Distribuição: {first_feat}')
axes[0].set_xlabel(first_feat)

corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.2, ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('Matriz de Correlação (features)')

plt.tight_layout()
plt.show()

Na exploração inicial, percebemos alguns pontos importantes. A base tem muitas variáveis com comportamento parecido, o que já indica colinearidade entre algumas features, e a matriz de correlação confirmou isso com vários pares fortes fora da diagonal principal. Também olhamos o balanceamento do target, porque em problema médico tabular isso costuma vir bem desbalanceado; aqui existe diferença, mas não é tão extrema (357 benignos e 212 malignos). No fim, o problema fica bem caracterizado como classificação supervisionada em dados tabulares, com alvo binário (benigno vs. maligno), então o próximo passo faz sentido ser a preparação dos dados.

### 2. Preparação dos Dados
*Pré-processamento e divisão treino/validação/teste.*

Aqui vamos manter a sequência correta para evitar vazamento de dados, fazemos o split em treino, validação e teste. Depois, ajustamos a redução por correlação usando só o treino e aplicamos as mesmas colunas em validação e teste. Por fim, padronizamos com `StandardScaler` ajustado apenas no treino.

In [ ]:
def fit_corr_feature_selector(X_train_df, threshold=0.95):
    feature_cols = X_train_df.columns.tolist()

    corr_feat = X_train_df.corr().abs()
    upper_feat = corr_feat.where(np.triu(np.ones(corr_feat.shape), k=1).astype(bool))

    high_pairs_095 = (
        upper_feat.stack()
        .reset_index()
        .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'corr_abs'})
    )
    high_pairs_095 = high_pairs_095[high_pairs_095['corr_abs'] >= threshold].reset_index(drop=True)

    adj = {f: set() for f in feature_cols}
    for _, row in high_pairs_095.iterrows():
        a = row['feature_1']
        b = row['feature_2']
        adj[a].add(b)
        adj[b].add(a)

    visited = set()
    components = []
    for f in feature_cols:
        if f in visited:
            continue
        stack = [f]
        comp = set()
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            comp.add(node)
            stack.extend(adj[node] - visited)
        components.append(comp)

    keep_features = [sorted(comp)[0] for comp in components]
    return sorted(keep_features), high_pairs_095

print('Função de redução por correlação pronta (fit no treino).')

Resumo: para evitar vazamento de dados, a redução por correlação passou a ser ajustada apenas no conjunto de treino. Depois aplicamos esse mesmo subconjunto de colunas em validação e teste. Em seguida, padronizamos com `StandardScaler`, com `fit_transform` só no treino e `transform` em validação/teste. Assim a avaliação fica metodologicamente mais correta.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, target_numeric, test_size=0.15, random_state=42, stratify=target_numeric
)

val_ratio_adjusted = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio_adjusted, random_state=42, stratify=y_temp
)

print(f'Features antes da redução (treino): {X_train.shape[1]}')

keep_features, high_pairs_095 = fit_corr_feature_selector(X_train, threshold=0.95)

X_train = X_train[keep_features].copy()
X_val = X_val[keep_features].copy()
X_test = X_test[keep_features].copy()

print(f'Features após redução (fit no treino): {len(keep_features)}')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'Shapes -> treino: {X_train.shape}, validação: {X_val.shape}, teste: {X_test.shape}')

Resumo: primeiro separamos treino, validação e teste com estratificação para manter a proporção das classes parecida nos três conjuntos. Depois padronizamos as variáveis com `StandardScaler`. O `fit_transform` fica só no treino porque é nele que o scaler aprende média e desvio padrão; em validação e teste usamos apenas `transform`, reaplicando essa mesma escala. Fazemos assim para evitar vazamento de dados e manter a avaliação mais confiável.

### 3. Treinamento + Validação

Nesta etapa comparamos duas configurações de MLP. Não estamos apenas treinando, em cada época calculamos métricas de treino e também de validação, e escolhemos o melhor estado do modelo com base na perda de validação. Vamos separar definição do modelo/funções e execução do treino + validação.

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

input_dim = X_train.shape[1]
num_classes = 2

train_params = {
    'epochs': 15,
    'batch_size': 32,
    'learning_rate': 1e-1,
}

mlp_configs = {
    'Config A': {
        'hidden_dims': [2],
        'dropout': 0.1,
        'weight_decay': 0.0,
    },
    'Config B': {
        'hidden_dims': [16, 8],
        'dropout': 0.2,
        'weight_decay': 1e-3,
    }
}

print(f"Input dim: {input_dim} | Classes: {num_classes}")
print("Parâmetros de treino:")
display(pd.DataFrame([train_params]))

print("Configurações comparadas:")
display(pd.DataFrame(mlp_configs).T)

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.0, num_classes=2):
        super().__init__()
        layers = []
        prev = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=train_params['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=train_params['batch_size'], shuffle=False)


def train_and_test_model(_, config):
    model = MLPModel(
        input_dim=input_dim,
        hidden_dims=config['hidden_dims'],
        dropout=config['dropout'],
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=train_params['learning_rate'],
        weight_decay=config['weight_decay']
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    best_val_loss = float('inf')
    best_state = None

    for _ in range(train_params['epochs']):
        model.train()
        train_loss_sum, train_correct, train_total = 0.0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * yb.size(0)
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == yb).sum().item()
            train_total += yb.size(0)

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)

                val_loss_sum += loss.item() * yb.size(0)
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == yb).sum().item()
                val_total += yb.size(0)

        epoch_train_loss = train_loss_sum / train_total
        epoch_val_loss = val_loss_sum / val_total
        epoch_train_acc = train_correct / train_total
        epoch_val_acc = val_correct / val_total

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history, best_val_loss

In [ ]:
mlp_runs = {}
for cfg_name, cfg in mlp_configs.items():
    model, history, best_val = train_and_test_model(cfg_name, cfg)
    mlp_runs[cfg_name] = {
        'model': model,
        'history': history,
        'best_val_loss': best_val,
    }

resumo_validacao = pd.DataFrame([
    {'config': name, 'best_val_loss': run['best_val_loss']}
    for name, run in mlp_runs.items()
]).sort_values('best_val_loss').reset_index(drop=True)

mlp_best_name = resumo_validacao.loc[0, 'config']
mlp_best_model = mlp_runs[mlp_best_name]['model']

display(resumo_validacao)
print(f"Melhor configuração na validação: {mlp_best_name}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cfg_name, run in mlp_runs.items():
    axes[0].plot(run['history']['train_loss'], label=f"{cfg_name} - train")
    axes[0].plot(run['history']['val_loss'], linestyle='--', label=f"{cfg_name} - val")

axes[0].set_title('Loss por época')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)

for cfg_name, run in mlp_runs.items():
    axes[1].plot(run['history']['train_acc'], label=f"{cfg_name} - train")
    axes[1].plot(run['history']['val_acc'], linestyle='--', label=f"{cfg_name} - val")

axes[1].set_title('Acurácia por época')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

### 4. Métricas
*Resultados quantitativos: Acurácia, Precisão, Recall, F1 e Matriz de Confusão.*


In [ ]:
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_np = y_test.values

results = []
conf_mats = {}

for cfg_name, run in mlp_runs.items():
    model = run['model']
    model.eval()

    with torch.no_grad():
        logits = model(X_test_tensor)
        y_pred = torch.argmax(logits, dim=1).cpu().numpy()

    acc = accuracy_score(y_test_np, y_pred)
    prec = precision_score(y_test_np, y_pred, average='binary', zero_division=0)
    rec = recall_score(y_test_np, y_pred, average='binary', zero_division=0)
    f1 = f1_score(y_test_np, y_pred, average='binary', zero_division=0)
    cm = confusion_matrix(y_test_np, y_pred)

    results.append({
        'config': cfg_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
    })
    conf_mats[cfg_name] = cm

results_df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
results_df_fmt = results_df.copy()
for col in ['accuracy', 'precision', 'recall', 'f1']:
    results_df_fmt[col] = results_df_fmt[col].map(lambda x: f"{x:.4f}")

print('Resumo de métricas no teste:')
display(results_df_fmt)

print('Matrizes de confusão (tabelas):')
for cfg_name, cm in conf_mats.items():
    print(f"\n{cfg_name}")
    display(pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1']))

### 5. Análise Crítica

Pelos resultados, as duas configurações tiveram desempenho aceitável, com vantagem da Config B. Na validação, ela ficou com menor perda (`best_val_loss = 0.030271` contra `0.043761` da Config A), e no teste também terminou melhor (`accuracy = 0.9884` e `f1 = 0.9841`, contra `0.9767` e `0.9688`). Nenhuma configuração passou de 99%, mas o resultado geral parece aceitável.

Também é importante considerar o tamanho do dataset. A entrada é pequena e, depois da divisão, a validação fica menor ainda. Nesse contexto, com um ajuste razoável de arquitetura e regularização, o modelo tende a errar pouco porque há menos casos e o padrão do problema é mais direto. Além disso, a redução de features provavelmente ajudou a diminuir ruído e facilitou a convergência.

Uma MLP é uma escolha coerente aqui porque o dado é tabular, numérico, não tem dados faltantes (não validei, mas é dito na documentação do [dataset](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)) e já vem com features estruturadas. Não há estrutura espacial de imagem nem relação explícita de grafo para justificar outro tipo de arquitetura.

Por fim, em problema de diagnóstico não dá para olhar só acurácia. Aqui não é o caso de uma distribuição extremamente desigual, mas a validação inicial foi feita justamente pensando nesse tipo de risco, já que ele é comum em bases médicas. Dependendo da distribuição das classes, é possível ter acurácia alta e ainda assim falhar em casos malignos, que são os mais críticos. Por isso, além da acurácia, a avaliação com precisão, recall, F1 e matriz de confusão foi necessária para enxergar melhor o risco real dos erros.

---
## Questão 2: CNN - Fashion-MNIST
**Objetivo:** Automatizar classificação de vestuário.


### 1. Descrição do Problema
*Descreva aqui a contextualização do problema para o dataset Fashion-MNIST.*


In [ ]:
# TODO: Carregamento do dataset (Fashion-MNIST)
pass


### 2. Preparação dos Dados
*Processo de normalização dos pixels, formatação (C, H, W) e divisão dos conjuntos.*


In [ ]:
# TODO: Pre-processamento
pass


### 3. Arquitetura Escolhida
*Definição e justificativa das camadas convolucionais (filtros, pooling) e Fully Connected. Incluir duas configurações.*


In [ ]:
# TODO: Implementação da CNN (mínimo 2 configurações)
pass


### 4. Configuração de Treinamento
*Loss function, otimizador, épocas, batch size.*


In [ ]:
# TODO: Loop de treino e exibição dos gráficos (loss / accuracy)
pass


### 5. Métricas e Avaliação
*Erros e acertos (plotar/mostrar os labels previstos vs reais nas imagens) e demais métricas de interesse.*


In [ ]:
# TODO: Métricas e visualizações (erros/acertos)
pass


### 6. Análise Crítica
*Discussão sobre o impacto dos filtros, do pooling e dos pesos compartilhados. Sinais de over/underfitting, e comparação entre as as duas configurações.*


In [ ]:
pass


---
## Questão 3: GAN - MNIST
**Objetivo:** Gerar novas amostras plausíveis (não classificar).


### 1. Descrição do Problema
*Contextualização e explicação da tarefa de geração de novas amostras.*


In [ ]:
pass


### 2. Preparação dos Dados
*Pré-processamento, normalização adequada para GANs e criação de dataloaders.*


In [ ]:
pass


### 3. Arquitetura Escolhida
*Definição do Gerador e do Discriminador. Incluir explicações sobre a estrutura de cada um e o vetor latente. Criar duas configurações.*


In [ ]:
pass


### 4. Configuração de Treinamento
*Definição dos otimizadores (geralmente um para cada rede), loss functions e detalhamento do vetor de ruído.*


In [ ]:
pass


### 5. Métricas (Avaliação/Geração)
*Registro visual das amostras geradas em diferentes momentos: início, meio e fim do treinamento.*


In [ ]:
# TODO: Plot das amostras geradas ao longo das épocas
pass


### 6. Análise Crítica
*Discussão sobre fenômenos clássicos da GAN: instabilidade de treino e mode collapse. Comparar as configurações.*


In [ ]:
pass


---
## Questão 4: GNN - Cora
**Objetivo:** Classificar artigos científicos em rede de citações.


### 1. Descrição do Problema
*Compreensão da base Cora e do problema de classificação de nós em grafos.*


In [ ]:
pass


### 2. Preparação dos Dados
*Carregamento do dataset Cora, divisão das máscaras de treino/teste/val, e representação do grafo (matriz de adjacência (A) e features (X)).*


In [ ]:
pass


### 3. Arquitetura Escolhida
*Implementação de Graph Convolutional Network (GCN) e definição dos nós/arestas. (Implementar duas configurações de hiperparâmetros/camadas).*


In [ ]:
pass


### 4. Configuração de Treinamento
*Loss e otimizadores.*


In [ ]:
pass


### 5. Métricas
*Resultados na tarefa de classificação de nós (Acurácia, etc).*


In [ ]:
pass


### 6. Análise Crítica
*Discussão sobre o mecanismo de propagação de informações entre vizinhos. Resultados e eventuais overfitting.*


In [ ]:
pass


---
## Questão 5: Escolha de Arquitetura
**Objetivo:** Reflexão sobre a prática do cientista de dados.


### Reflexão e Comparativos
- MLP vs CNN
- CNN vs GAN
- MLP vs GNN
  - Discutir custo computacional, interpretabilidade e estrutura do dado.
*Escreva sua análise teórica aqui.*
